# 14 · Where visitors see each species: a zoom-aware heat map

Notebooks 06–13 found that GBIF sightings can't support **habitat** maps: the models learn where visitors go, for every species. For a safari explorer, that's exactly the useful question: **where do visitors actually see each species?** This map answers it directly from the sightings, with no model.

**What the colour means:** of the animals visitors logged around a place, the share that were the chosen species. In other words, the chance that a sighting reported there is that species.
- It's **effort-adjusted.** Busy places don't glow just because many people pass through; Seronera has hundreds of records but a low lion share.
- **Colour only appears where the share is above the species' average.** A glow means "visitors see it here more than usual".

**Recalculated on every zoom and pan:** the share is recomputed in the browser from the records around each point on screen. The neighbourhood follows the zoom: about 18 km zoomed out to the whole area, about 2 km around Seronera, a few hundred meters zoomed fully in. Distances are in UTM meters (ADR-0001).

**Honest about thin data:**
- Where few records are nearby, the share is pulled toward the species' average (`prior_records`) and the colour fades (`confidence_records`).
- With almost no records, nothing is drawn.
- Clicking any point gives the numbers behind the colour, with a range.

Data: GBIF mammal sightings 2017–2025 (cached, from 11). No Earth Engine.

In [1]:
import buffalo_data as bd
import buffalo_validation as bv
import sightings_map as sm
from buffalo_data import BuffaloConfig

config = BuffaloConfig(cache_dir="cache")
records = bd.fetch_mammal_records(config)
species = {name: sp["speciesKey"] for name, sp in bv.SAFARI_SPECIES.items()}

heat_map = sm.sightings_heat_map(records, species, config.bounds, zoom_start=9, location=[-2.45, 34.85])
heat_map

mammal_records: loaded from cache\buffalo_mammal_records_8be5ffc111.parquet
mammal records in box: 15873


**Using it:**
- **Pick a species** in the box (top right). The legend shows its average share and the top of its colour scale, and how far each point is averaged at the current zoom.
- **Zoom in:** the shares are recomputed over a smaller neighbourhood, so broad glows split into the specific spots where the species is logged.
- **Click** anywhere for the numbers, e.g. *"Of ~30 animals logged within ~2.3 km, about 22 % were lion (range 9–35 %)."* A wide range means few records; treat that glow with care.

**What it doesn't say:**
- **Where a species lives,** or how many there are. It describes what visitors reported, on the roads they drive. A low share can also mean other animals dominate the logs there, for example wildebeest during the migration.
- **Anything about unvisited areas:** no records, no colour.
- **Season:** all months 2017–2025 are pooled. A species that moves seasonally gets averaged over the year.

**Settings** (arguments of `sm.sightings_heat_map`):

| setting | default | effect |
|---|---|---|
| `kernel_px` | 30 | averaging radius in screen pixels: larger = smoother, broader glows |
| `min_sigma_m` | 250 | never average over less than this; about the precision of GBIF coordinates |
| `prior_records` | 10 | how strongly thin estimates are pulled toward the species' average |
| `confidence_records` | 10 | colour fades as n / (n + this): few records stay faint |
| `min_visible` | 0.15 | shares less than 15 % of the way from average to the top aren't drawn |